# Generation of scene

## Imports

In [2]:
import numpy as np
from robotblockset.tools import get_rbs_path, print_xml, print_xml_for_console

from robotblockset.transformations import rot_x, rot_y, rot_z, map_pose

import mujoco 
from robotblockset.mujoco.tools_mjcf import print_body_tree, actuators_for_joint

## Initialization

In [ ]:
ROBOT = "panda_no_hand"
ROBOT_NAME = "panda"
GRIPPER = "panda_hand"
MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/"
CAMERA = "realsense_d435i"
TOOL = "calibration_plate"
PLATE = "charuco"

## Generation of scene

### Basic scene

In [4]:
scene=mujoco.MjSpec.from_file(MODEL_PATH + "scene_high_resolution.xml")
scene.copy_during_attach = True
print_body_tree(scene.worldbody, scene)
# print_xml(scene.to_xml())

Body Tree for "world"
-Target


### Attach robot

Load robot specification

In [ ]:
robot_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{ROBOT}.xml")
robot_spec.copy_during_attach = True
print_body_tree(robot_spec.worldbody, robot_spec)
robot_ctrl = np.array(robot_spec.keys[0].ctrl)
robot_qpos = np.array(robot_spec.keys[0].qpos)
# print_xml(robot_spec.to_xml())

Body Tree for "world"
-base
--link0
---link1 (Joints: joint1-HINGE[Actuator: pos_joint1])
----link2 (Joints: joint2-HINGE[Actuator: pos_joint2])
-----link3 (Joints: joint3-HINGE[Actuator: pos_joint3])
------link4 (Joints: joint4-HINGE[Actuator: pos_joint4])
-------link5 (Joints: joint5-HINGE[Actuator: pos_joint5])
--------link6 (Joints: joint6-HINGE[Actuator: pos_joint6])
---------link7 (Joints: joint7-HINGE[Actuator: pos_joint7])
----------flange


Attach robot to ground

In [ ]:
attachment_frame = scene.worldbody.add_frame(pos=[0, 0, 0], axisangle=[0, 0, 1, 0 * np.pi / 2])
attachment_frame.attach_body(robot_spec.body("base"), f"{ROBOT_NAME}_")
scene.body(f"{ROBOT_NAME}_base").name = ROBOT_NAME
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange


Exclude contacts between robot base and table

In [ ]:
# scene.add_exclude(bodyname1=f"{ROBOT_NAME}", bodyname2="ground")


### Attach gripper to robot

Load gripper specification, attach gripper to robot flange, and redefine robot TCP

In [ ]:
if GRIPPER is not None:
    gripper_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{GRIPPER}.xml")
    gripper_spec.copy_during_attach = True
    if len(gripper_spec.keys) > 0:
        gripper_ctrl = np.array(gripper_spec.keys[0].ctrl)
        gripper_qpos = np.array(gripper_spec.keys[0].qpos)
    else:
        gripper_ctrl = np.array([])
        gripper_qpos = np.array([])   
    attachment_frame = scene.body(f"{ROBOT_NAME}_flange").add_frame(pos=[0, 0, 0], quat=[0.9238795, 0, 0, -0.3826834])
    attachment_frame.attach_body(gripper_spec.body("hand"), f'{ROBOT_NAME}_tool_')
    scene.delete(scene.site(f"{ROBOT_NAME}_TCP"))
    scene.site(f"{ROBOT_NAME}_tool_hand_TCP").name = f"{ROBOT_NAME}_TCP"
    print_body_tree(scene.worldbody, scene)



Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_tool_hand
------------panda_tool_left_finger (Joints: panda_tool_finger_joint1-SLIDE)
------------panda_tool_right_finger (Joints: panda_tool_finger_joint2-SLIDE)


### Attach tool to gripper

Load tool and attach it to the gripper

In [9]:
if TOOL is not None:
    tool_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{TOOL}.xml")
    tool_spec.copy_during_attach = True
    print_body_tree(tool_spec.worldbody, tool_spec)


Body Tree for "world"
-calib_charuco
-calib_checker


In [ ]:
if TOOL is not None:
    scene.site("panda_TCP").pos = scene.site(f"{ROBOT_NAME}_TCP").pos + [0, 0, tool_spec.geom(f"{PLATE}_pattern").size[1]]
    scene.site("panda_TCP").quat = rot_y(np.pi / 2)    
    attachment_frame = scene.body(f"{ROBOT_NAME}_tool_hand").add_frame(pos=scene.site(f"{ROBOT_NAME}_TCP").pos, quat=scene.site(f"{ROBOT_NAME}_TCP").quat)
    attachment_frame.attach_body(tool_spec.body(f"calib_{PLATE}"), "plate_")

print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_tool_hand
------------panda_tool_left_finger (Joints: panda_tool_finger_joint1-SLIDE)
------------panda_tool_right_finger (Joints: panda_tool_finger_joint2-SLIDE)
------------plate_calib_charuco


### Add cameras

Load camera specification

In [11]:
if CAMERA is not None:
    camera_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{CAMERA}.xml")
    camera_spec.copy_during_attach = True
    camera_spec.body("cam").mocap=0
    print_body_tree(camera_spec.worldbody, camera_spec)

Body Tree for "world"
-cam


Attach camera

In [12]:
if CAMERA is not None:
    attachment_frame = scene.worldbody.add_frame(pos=[1, 0, 0.6], axisangle=[0, 0, 1, 2 * np.pi / 2])
    attachment_frame.attach_body(camera_spec.body("cam"), "cam_")
    print_body_tree(scene.worldbody, scene)


Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_tool_hand
------------panda_tool_left_finger (Joints: panda_tool_finger_joint1-SLIDE)
------------panda_tool_right_finger (Joints: panda_tool_finger_joint2-SLIDE)
------------plate_calib_charuco
-cam_cam


### Prepare MJCF XML file

Redefine MJCF keys. Delete all keys and define Key 0 as the initial configuration

In [13]:
scene.keys

[]

In [14]:
_tmp = scene.to_xml()
while len(scene.keys)>1:
    scene.delete(scene.keys[-1])
for k in scene.keys:
    k.ctrl = np.concatenate([robot_ctrl, gripper_ctrl, np.zeros(len(scene.actuators) - len(robot_ctrl) - len(gripper_ctrl))])

In [15]:
scene.body("cam_cam").mocap=1

Save MJCF scene to XML

In [16]:
scene.modelname = f"calibration_{PLATE}_scene"
scene.option.timestep = 0.001
with open(MODEL_PATH + f"{scene.modelname}.xml", "w") as f:
    f.write(scene.to_xml())